# FENCINGLY · Parte 1 — extracción a duración completa y cadencia nativa

**Qué hace este cuaderno.** Extrae cada clip **entero** (`f0 = 0`, `f1 = n_frames`) a la
**cadencia del contenedor**, sin remuestrear y sin recortar, y escribe un `npz/index.csv` de
119 columnas que el lector real de la aplicación sabe leer.

**Qué NO hace.** No entrena, no mide y no decide la rodaja de entrenamiento: eso es el paso 5
y es un **mando aparte**, porque extraer y entrenar son dos decisiones.

---

## Lo que hay que saber antes de ejecutar

**La extracción nunca remuestreó a 30 fps.** `pipeline.stage_save` graba
`time_s = (f0 + arange(n)) / lr["fps"]` con el fps del contenedor. Medido sobre 24 NPZ: el
desvío del eje respecto de `1/fps` nativo es **0,0 µs**, con fps de **1,661 a 60,003**.
El remuestreo a 30 fps vive en `sample_time.TARGET_FPS = 30.0`, y lo aplica `dataset.prepare`
al construir el **almacén de entrenamiento**. Ahí sí se borra el **44,4 %** de los fotogramas
de los clips por encima de 30,5 fps. **Son dos sitios distintos y hay que tocar los dos.**

**Lo que la extracción sí recortaba es la ventana:** `PRE_S, POST_S = 4.0, 0.3`. Este cuaderno
la abre a todo el clip pidiendo `pre_s = post_s = 30 s` y `use_clock=False`. Con
`use_clock=True` el inicio salta al *allez* y vuelve a recortar la cabeza.

**Por qué desde el fotograma 0.** No es para desbloquear la escala del *allez* — con `f0 = 0`,
`calibration.allez_scale` rechaza por `window_not_at_clock_start`. Es porque **es la única
configuración que permite elegir el ancla después**: rodaja en el tocado para el almacén y las
medidas, vista en el *allez* para la escala. Las poses del *allez* quedan dentro del NPZ.

### Coste medido (24 clips, RTX 3060, `imgsz` 960, 1 proceso)

| | valor |
|---|---|
| fotogramas guardados | **×2,61** (3.451 → 8.990) |
| tamaño | **159,7 bytes/fotograma**; los 34.374 clips caben en **1,65–1,99 GiB** |
| tiempo | **13,41 s/clip** mediana; **derivado**: ≈63,7 h con 4 procesos, ≈40–45 h con `max_gap_redetect=12` |
| resolución | nativa 720p 78,1 % · 1080p 20,6 % · 480p 1,4 %. **No hay 4K** |

**El disco no es el problema.** El cuello es **CPU de descodificación** (`prepare`+`save` = 27 %
del tiempo), así que una GPU más rápida **no escala linealmente**: en Colab hay 2-8 núcleos y
ahí está el límite.

## Paso 0 · Puertas

**Levantan, no avisan.** Versiones exactas de producción, huella del productor y parche de
ventana.

**Dos raíces, y no son la misma.** `RAIZ_CODIGO` es el árbol con el que se **extrae** (puede
ser una copia congelada). `RAIZ_MEDIDA` es el árbol **vivo**, el único con el que la huella
significa algo. Pedirle la huella a la copia congelada da `fd2421b3…` en vez de `3e1dbe0a…`
**sin ningún error** — es el punto 19 de `AGENTS.md`, y esta puerta lo cazó la primera vez que
se ejecutó porque los dos argumentos están separados.

**`tqdm` no se da por hecho.** No está instalado en `.venv` ni en `.venv-gpu`, y
`pipeline.process_many` hace `try: from tqdm.auto import tqdm … except: bar = None`, así que
**en local la barra no existe y no avisa**. `barra.py` usa tqdm si está y un respaldo propio
con el mismo API si no.

In [ ]:
import os, sys, json
from pathlib import Path

# --- rutas: ajustar solo este bloque ---
BASE        = Path("/content/FENCINGLY")          # raiz del paquete en Colab
RAIZ_CODIGO = BASE / "codigo_congelado"           # con el que se EXTRAE
RAIZ_MEDIDA = BASE / "webapp"                     # arbol VIVO: la huella sale de aqui
WS          = BASE / "ws_completo"                # workspace de salida
DIR_MP4     = BASE / "mp4"                        # o el Drive montado
MODELO_INDEX = BASE / "modelo_index.csv"          # un index.csv de 119 columnas QUE YA CARGA
PESOS       = BASE / "yolo11m-pose.pt"
MANIFIESTO  = BASE / "manifest_foil.csv"
EXTRA       = []          # ficheros JSON/CSV o URLs de JSON con clips nuevos

for p in (str(RAIZ_CODIGO), str(BASE / "catalogo")):
    if p not in sys.path:
        sys.path.insert(0, p)

import pasos
ENT = pasos.paso0_puertas(RAIZ_CODIGO, raiz_medida=RAIZ_MEDIDA, estricto=True)

## Paso 1 · Manifiesto, y cómo añadirle clips nuevos

**Extensión:** `EXTRA` acepta ficheros `.json` / `.jsonl` / `.csv` **o una URL de JSON**. Cada
fuente se valida contra el esquema mínimo (`clip_id`); la que no lo cumple **se rechaza con su
motivo** y no aborta la corrida. En un `clip_id` duplicado se queda **la primera**, igual que
`AnalysisRepository` con el orden de `--npz-dir`.

**Filtro de florete, explícito.** `weapon == "foil"`, comprobado en la tabla. Los clips **sin
arma se rechazan**, no se asumen florete — el fichero se llame `manifest_foil.csv` o como sea.
Si el manifiesto no trae columna `weapon`, la celda **levanta**.

In [ ]:
MAN = pasos.paso1_manifiesto(MANIFIESTO, extra=EXTRA, arma="foil")
MAN.head()

## Paso 2 · Estado: la base de datos de qué está hecho

**El estado es `index.csv` ∩ `.npz` en disco. Nunca un contador.**

Por qué: `AnalysisRepository.index()` **ignora en silencio** un directorio sin `index.csv`, y
eso dejó **689 clips invisibles**. Y `valid_id` exige `[A-Za-z0-9_-]{1,200}`, así que un
`clip_id` no-ASCII se cae del índice **sin error** (13 clips conocidos: `Vicuña`, `Kauai‘Ō‘ō`,
`Vendéen`). Es un hueco **conocido y aceptado**: no relajar el validador.

La celda imprime el conteo en cada eslabón, para ver **cuál** deja caer clips:

- **`.npz` sin fila** → invisibles para el lector. Se arregla **reindexando, no reextrayendo**.
- **fila sin `.npz`** → el lector fallará al abrirlo. No se emite fila sin fichero.

In [ ]:
EST = pasos.paso2_estado(MAN, WS)
print(f"\na extraer en esta corrida: {len(EST['pendiente'])}")

## Paso 3 · Extracción

**Comprueba por clip que `f0 == 0` y `f1 == n_frames`.** Si falla, la fila queda marcada
`completo=False`; no es un aviso que se lee después.

**`max_gap_redetect = 12`** en vez de 25. `fill_gaps` reejecuta la red por cada fotograma sin
tirador vía `redetect_lowres`/`redetect_roi`, y con duración completa los primeros planos de
repetición entran en la ventana. Medido sobre 24 clips: **235,2 s → 16,9 s (−92,8 %)** por
**−0,4 pp** de cobertura mediana, y en el clip patológico **141 s → 7,9 s**.

De los **117** puntos perdidos, **0 caen en la ventana de guardia** `[−0,5, +0,3] s`
(criterio declarado antes de medir). **Salvedad escrita al lado:** 114 de los 117 caen a
**−0,8 s**, dentro de la fase de preparación, de donde salen `frac_adv_prep` y
`prep_detenida`. `prep_detenida` está medida con **AUC 0,4997**, así que para prioridad no
cuesta nada medible; para estadística descriptiva sí.

`pose.py` **no está** en `MEASUREMENT_SOURCES` ni en `MOTION_SOURCES`, y `max_gap_redetect` es
un argumento de función: el valor **no mueve ninguna huella**.

**Dos barras**: la corrida y el clip en curso, con ritmo y tiempo restante.

**Aviso de RAM:** el array `(T,H,W,3)` uint8 a duración completa llega a **5,29 GiB** en el
peor clip medido (p95 2,53 GiB). `ram_max_gib` salta el clip y **lo reporta** en vez de morir.

In [ ]:
from fencingly_v2 import pipeline as PL, pose as P, video as V
from fencingly_v2 import colab_workflow as CW

extractor = CW.create_extractor(str(PESOS), pasos.PLAN_POSE)
extractor.batch = pasos.PLAN_POSE["pose_batch"]
print("modelo cargado: imgsz", extractor.imgsz, "lote", extractor.batch)

# clip_id -> ruta del mp4. El nombre del fichero ES el clip_id.
RUTAS = {p.stem: str(p) for p in Path(DIR_MP4).rglob("*.mp4")}
print(f"mp4 localizados: {len(RUTAS)}")

RES = pasos.paso3_extrae(EST["pendiente"], RUTAS, WS, extractor, PL, P, V,
                         max_gap=pasos.MAX_GAP_REDETECT, ram_max_gib=6.0)

## Paso 4 · Índice, contado por el lector real

Se escribe con el **orden de columnas de un índice que ya carga** (119), y se emite fila
**solo si su `.npz` está en disco**. Después se cuenta con `AnalysisRepository.index()`, **no
con las filas del CSV**: eso es lo que falló anoche con 689 clips.

In [ ]:
import pandas as pd
IDX = pasos.paso4_indice(RES, WS, MODELO_INDEX, manifiesto=MAN)

# En una RE-EJECUCION sin pendientes, `paso4_indice` no reescribe el indice (y hace bien:
# no lo sobreescribe con nada). Entonces `IDX` viene vacio y verificar contra el daria un
# OK que no comprueba nada. Asi que lo esperado se toma del indice EN DISCO.
if len(IDX):
    esperados = set(IDX["clip_id"].astype(str))
else:
    f = WS / "npz" / "index.csv"
    esperados = (set(pd.read_csv(f, usecols=["clip_id"], low_memory=False)["clip_id"].astype(str))
                 if f.is_file() else set())
    print(f"  re-ejecucion: se verifica contra el index.csv en disco ({len(esperados)} clips)")

VER = pasos.paso4b_verifica_lector(WS, RAIZ_MEDIDA, esperados)
assert esperados, "no hay nada que verificar: ni index.csv nuevo ni en disco"
assert VER.get("ok") is not False, "hay clips INVISIBLES para el lector: revisar antes de seguir"

## Paso 5 · Rodaja de entrenamiento — **mando aparte, y no toca los NPZ**

**`store.T_MAX = 135` y el almacén alinea por la DERECHA.** Un clip de 500 fotogramas a 50 fps
se queda con los últimos 135 = **2,70 s, el 27 % del clip**, y tira el resto **sin dar error**.
Y `dataset.prepare` rehace el eje a 30 fps, que en los clips rápidos borra el **44,4 %**.
**Ninguna de las dos cosas avisa.**

Por eso esto es un paso separado y con `aplicar=False`: **imprime lo que se perdería** con el
anclaje y el `t_max` que elijas, y no modifica nada. Anclajes: `"tocado"` (para el almacén y
las medidas) o `"allez"` (para la escala).

In [ ]:
ROD = pasos.paso5_rodaja(WS, t_max=135, anclaje="tocado", pre_s=4.0, post_s=0.3, aplicar=False)
# y con el otro ancla, para comparar:
_ = pasos.paso5_rodaja(WS, t_max=600, anclaje="allez", aplicar=False)

## Paso 6 · Tabla por fotograma (Parquet)

Una fila por `(clip_id, frame_idx)`. **`NaN` significa indefinido, nunca cero**: un cero es una
posición, un `NaN` es «no se midió».

**`src_fps` y el ratio `fps/src_fps` viajan los dos como columnas, y son lo más importante de
esta tabla.** `stage_save` calcula `src_fps = fps · src_ratio` cuando `dup_mode == "regular"`:
la cadencia **real** de la fuente. Sin ellos, «50 fps» y «60 fps con origen a 15 fps» se tratan
igual **y no lo son**.

**Y hay un artefacto mayoritario que no estaba en ninguna lista.** Medido sobre los 15.452
clips del índice vivo (`dup_mode` presente en el 100 %), por fps **nominal redondeado**:

| nominal | clips | patrón regular | ratio ≥ 2 |
|---|---:|---:|---:|
| 25 | 2.894 | 10 (0,3 %) | **0** |
| **30** | **11.211** | **8.503 (75,8 %)** | 35 |
| **50** | 649 | **0 (0,0 %)** | **0** |
| 60 | 170 | 66 (38,8 %) | **19 (11,2 %)** |

- **El problema es la banda de 30, que es la mayoritaria:** 8.503 clips con **telecine 25→30**.
  `dup_rate` mediana **0,164**, `dup_period` **6,0 fotogramas exactos**, `src_fps` mediana
  **25,0**. **Uno de cada seis fotogramas es un duplicado**, y un duplicado da **velocidad cero
  en un paso y doble en el siguiente**.
- **La cámara lenta es residual: 54 clips de 15.452, el 0,35 %.** Ratios 2,0 · 2,4 · 4,0 · 6,0
  · **12,0**. Un ratio de 12 son **once fotogramas de velocidad cero seguidos de uno con
  velocidad doce veces mayor** — *eso* es lo que rompe una derivada, no la cadencia nominal.
  Sedes: eurochamps2022 (19), tbilisifoil2023 (19), worldchamps2022 (8), serbiafoil2022 (4).
- **A 50 fps: cero clips con patrón regular.** Es lo que sostiene el argumento del paso de
  20 ms. A 60 fps el 38,8 % tiene duplicados pero solo **19 de 170** están multiplicados de
  verdad; el resto es conversión 50→60 a ratio 1,20, material real reempaquetado, **no lento**.

Tres consecuencias, y son obligatorias:

1. **`src_fps` y el ratio viajan siempre.**
2. **El suavizado va ANTES de derivar**, y su ventana se justifica **contra un duplicado cada
   seis fotogramas** —no contra la cámara lenta—: por debajo de ~7 fotogramas no absorbe el
   artefacto. Guardar **cruda y suavizada**, las dos.
3. **Toda medida de velocidad se reporta con `src_fps` al lado**, no solo con la nominal.

> **Aviso de denominadores.** El censo completo tiene **29.220 clips con fps** (20.938 a 30,
> 4.883 a 25, 1.547 a 50, 816 a 60), pero `dup_mode` está medido en **15.452 (52,9 % del censo,
> 100 % del índice vivo)**. **No multiplicar la tasa por el censo:** la muestra medida no es
> aleatoria, es la que se llegó a procesar, y el residuo está sesgado (la tasa de detección
> correlaciona **r = −0,861** con la fracción ya extraída). Citar las dos cifras con su n.

`escala_r` va **cruda y continua**, con su `NaN`. No es una bandera: ver el paso 7.

In [ ]:
TAB = pasos.paso6_tabla(WS, salida_parquet=WS / "metricas_por_fotograma.parquet")
TAB.head()

## Paso 7 · Huecos declarados

**La mitad del valor de este cuaderno.** Si algo no se puede sacar de vídeo de retransmisión a
25-60 fps con lo que hay, saberlo ahorra el intento.

In [ ]:
pasos.paso7_huecos()

---

## Riesgo abierto número uno: contaminación silenciosa por planos de repetición

`PeriodicSlipperyChipmunk` (50 fps, `tokyoolympics2020`) **abre con primeros planos de
repetición del tocado**, con un barrido de anillos olímpicos, y solo después pasa al plano en
vivo. La ventana de 4,3 s se quedaba con el plano en vivo; la duración completa mete la
repetición dentro.

**Y la cobertura de pose es 1,000 en L y en R.** El detector encuentra a los dos tiradores
porque **están ahí**, en otro encuadre y a otra escala, y **ninguna comprobación existente lo
marca**. Que la cobertura bajara era una alarma; aquí no hay ninguna. `cuts` reporta **1**
donde a la vista hay tres planos.

Las cuatro vías probadas y por qué ninguna caza:

| vía | resultado |
|---|---|
| `dup_rate` | ciego a 50 fps: mediana 0,000, cero `dup_mode="regular"` |
| `cuts` | sub-detecta: 1 contra ≥3 planos visibles |
| `reloj_parado_todo_el_clip` | >30,5 fps **20,0 %** [14,2, 27,4] vs ≤30,5 fps **20,5 %** [11,2, 34,5]. Solapan, −0,5 pp: no separa |
| escala aparente de caja | control `r_max = 1,58` contra umbral 1,60; clip limpio 1,32. **1,32 y 1,58 se solapan** |

**No se bajó el umbral a 1,55 para que el control pasara.** Un umbral elegido para que pase el
control es un umbral que mide el control — es el error del umbral heredado del reloj, al revés.

Y el primer intento es más instructivo que el segundo: normalizar por la mediana **del clip
entero**, cuando la repetición es la mitad del clip, **contamina el normalizador con lo que se
quiere detectar**. Es el mismo error de clase que el nulo circular preservando las columnas de
nivel.

## Confusor de sede en la cadencia

Los 904 clips de alta cadencia salen de **tres sedes**: `tokyoolympics2020` (637),
`tokyofoil2022` (190), `tbilisifoil2023` (77). Y los **154** clips de 50 fps con corte de plano
son **todos** de `tokyoolympics2020`, los 154.

**O sea: «alta cadencia» es en buena parte «Tokio 2020».** Cualquier comparación por cadencia
arrastra el confusor de sede, y alguien lo va a leer como un efecto de la cadencia. Es la
tercera vez en la sesión que un confusor de sede muerde.

## Y una máscara que no basta

Tapar `y > 0,82·H` **no es suficiente**: el marcador del árbitro con sus lámparas puede quedar
**por encima** de esa línea. Este cuaderno no recorta nada, pero cualquier consumidor de estos
NPZ que aplique esa máscara tiene que saberlo.